# 07 — Ensemble: blend, vaznli blend, stacking

Faza 7: Faza 5'da o'qitilgan uchta model (`lgbm_tuned`, `xgb`, `catboost`) OOF
ehtimolliklarini birlashtirib, yakka modeldan yaxshiroq natija olish mumkinmi
— tekshiramiz. Yordamchi funksiyalar `src/ensemble.py` da.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.config import load_config
from src.train import (
    prepare_train_test, get_numeric_feature_cols, get_catboost_feature_cols, prepare_catboost_frame,
    build_lgbm_model, build_xgb_model, make_lgbm_fit_kwargs, xgb_fit_kwargs, run_cv, RAW_CATEGORICAL_COLS,
)
from src.evaluate import load_oof, overall_log_loss, per_class_log_loss, is_improvement_significant
from src.ensemble import blend_proba, optimize_blend_weights, stacking_oof

cfg = load_config()
# groups=("categorical",) — Faza 5/6 da eng yaxshi natija bergan "baza" feature to'plami
# (log1p/nisbat/klinik ball qo'shimcha feature'lari daraxt modellari uchun CV'ni
# yomonlashtirgan edi, 03_experiments.ipynb §1). Yakuniy model AYNAN shu bilan o'qitilishi kerak.
train, test = prepare_train_test(cfg, groups=("categorical",))
comp_mask = (train["is_original"] == 0).to_numpy()
y_true = train.loc[comp_mask, cfg.data.target].to_numpy()
CLASSES = cfg.data.classes

MODEL_NAMES = ["lgbm_tuned", "xgb", "catboost"]
oof = {name: load_oof(name, cfg, comp_mask) for name in MODEL_NAMES}
for name, proba in oof.items():
    print(f"{name:12s} solo overall = {overall_log_loss(y_true, proba):.5f}")

13:25:56 | INFO    | src.data | load_raw: train=(15000, 19) test=(10000, 18)


13:25:56 | INFO    | src.data | load_original: (418, 19) (UCI, 6 null Stage)


13:25:56 | INFO    | src.data | merge_original: train=15000 + original=418 -> combined=15418


13:25:56 | WARNING | src.data | encode_target: 1 qator kutilmagan Status qiymati bilan tashlandi: ['Y']


lgbm_tuned   solo overall = 0.36288
xgb          solo overall = 0.36646
catboost     solo overall = 0.37894


## 1. Oddiy o'rtacha blend

Eng sodda usul — uchala modelning ehtimolliklarini teng vaznda o'rtachalash.

In [2]:
probas = [oof[n] for n in MODEL_NAMES]
equal_weights = np.full(len(probas), 1 / len(probas))
blend_equal = blend_proba(probas, equal_weights)

print(f"Oddiy o'rtacha blend: overall = {overall_log_loss(y_true, blend_equal):.5f}")
print(f"(solishtirish uchun eng yaxshi yakka model — lgbm_tuned: {overall_log_loss(y_true, oof['lgbm_tuned']):.5f})")

Oddiy o'rtacha blend: overall = 0.36574
(solishtirish uchun eng yaxshi yakka model — lgbm_tuned: 0.36288)


**Kutilmagan natija:** oddiy o'rtacha blend yakka `lgbm_tuned` dan **yomonroq**!
Sabab aniq — `catboost` (solo `0.379`) boshqa ikkisidan sezilarli yomonroq,
va uni teng vaznda qo'shish umumiy natijani pasaytiradi. Bu shuni ko'rsatadiki,
**"ko'proq model — har doim yaxshiroq blend" degani emas** — zaif modelni
teng vaznda qo'shish kuchli modelni "suyultiradi".

## 2. Vaznli blend — vaznlarni optimallashtirish

`scipy.optimize.minimize` (SLSQP) bilan CV log loss'ni minimallashtiradigan
vaznlarni topamiz (vaznlar manfiy emas, 1ga yig'iladi).

In [3]:
opt_weights = optimize_blend_weights(y_true, probas)
blend_opt = blend_proba(probas, opt_weights)

for name, w in zip(MODEL_NAMES, opt_weights):
    print(f"  {name:12s}: vazn = {w:.4f}")
print(f"\nVaznli blend: overall = {overall_log_loss(y_true, blend_opt):.5f}")

  lgbm_tuned  : vazn = 0.9006
  xgb         : vazn = 0.0994
  catboost    : vazn = 0.0000

Vaznli blend: overall = 0.36284


Optimallashtirish `catboost`ga **~0 vazn** berdi — u umuman foydali hissa
qo'shmayapti (boshqa ikkisi bilan yuqori korrelyatsiyalangan, lekin ular
orasida eng zaifi). Amalda bu 2-modelli (`lgbm_tuned` ~90%, `xgb` ~10%)
blend'ga aylandi. Natija (`~0.3628`) yakka `lgbm_tuned` dan (`0.36288`)
atigi `~0.00004` farq qiladi — **bu farq statistik jihatdan ahamiyatsiz**
(6-bo'limda tekshiramiz).

## 3. Stacking

Uchala modelning OOF ehtimolliklarini (`3 model x 3 sinf = 9` feature) kirish
sifatida olib, ularning ustiga LogisticRegression meta-model o'qitamiz.
Meta-model o'zi ham nested CV bilan baholanadi (leakage'ni oldini olish uchun).

In [4]:
oof_stack, fold_losses = stacking_oof(y_true, probas)
print("Fold log loss'lari:", [round(x, 5) for x in fold_losses])
print(f"Stacking: overall = {overall_log_loss(y_true, oof_stack):.5f}")

Fold log loss'lari: [0.40159, 0.37369, 0.38606, 0.38025, 0.38463]


Stacking: overall = 0.38525


**Stacking eng yomon natija berdi** — yakka modellarning barchasidan ham
pastroq! Sabab: 9 ta feature juda kuchli korrelyatsiyalangan (har bir model
ichida 3 ustun 1ga yig'iladi — chiziqli bog'liqlik), va asosiy modellar
(hammasi bir xil "baza" feature to'plamida o'qitilgan daraxt modellari)
juda o'xshash xatoliklar qiladi — meta-model qo'shimcha signal topa olmay,
faqat ortiqcha shovqin (5-fold ichida yana bir CV qatlami) qo'shadi.

## 4. Yakuniy taqqoslash va qaror

In [5]:
comparison = pd.DataFrame([
    {"usul": "lgbm_tuned (yakka, eng yaxshi)", "overall": overall_log_loss(y_true, oof["lgbm_tuned"])},
    {"usul": "oddiy o'rtacha blend (3 model)", "overall": overall_log_loss(y_true, blend_equal)},
    {"usul": "vaznli blend (optimallashtirilgan)", "overall": overall_log_loss(y_true, blend_opt)},
    {"usul": "stacking (LogisticRegression)", "overall": overall_log_loss(y_true, oof_stack)},
]).sort_values("overall").reset_index(drop=True)
comparison

,usul,overall
0,vaznli blend (optimallashtirilgan),0.362838
1,"lgbm_tuned (yakka, eng yaxshi)",0.362882
2,oddiy o'rtacha blend (3 model),0.365738
3,stacking (LogisticRegression),0.385246


In [6]:
# lgbm_tuned solo CV std (Faza 5/6 dan) va vaznli blend'ning taxminiy std'i
# (blend fold-fold hisoblanmagan, shuning uchun solo modelning std'idan foydalanamiz — taxminiy)
LGBM_STD = 0.00926
significant = is_improvement_significant(
    overall_log_loss(y_true, oof["lgbm_tuned"]), LGBM_STD,
    overall_log_loss(y_true, blend_opt), LGBM_STD,
)
print(f"Vaznli blend yakka lgbm_tuned'dan STATISTIK jihatdan ahamiyatli farq qiladimi? {significant}")

Vaznli blend yakka lgbm_tuned'dan STATISTIK jihatdan ahamiyatli farq qiladimi? False


### Qaror: **yakka `lgbm_tuned` model tanlandi**

Sabablari:
1. Vaznli blend statistik jihatdan yakka modeldan **ajratib bo'lmaydigan**
   darajada yaqin (`0.36284` vs `0.36288`) — farq CV shovqinidan ancha kichik.
2. Oddiy blend va stacking **yomonlashtirdi**.
3. Qo'shimcha murakkablik (3 modelni saqlash, blend vaznlarini boshqarish)
   hech qanday amaliy foyda bermaydi — **Occam ustarasi**: teng natija
   bo'lsa, soddaroq yechim afzal.

Bu — reja hujjatining 5-qoidasiga ("yomon natijali eksperimentlarni ham
jadvalda qoldiring") mos yana bir misol: ensemble g'oyasi o'zi mantiqiy
edi, lekin bu aniq muammoda (3 ta juda o'xshash daraxt modeli) foyda
bermadi — va buni **sinab ko'rmasdan bilib bo'lmas edi**.

## 5. Yakuniy submission generatsiyasi

Tanlangan yondashuv (`lgbm_tuned`) bo'yicha 5 ta fold modelini qaytadan
o'qitib, test.csv uchun bashorat qilamiz (5-fold bagging, `04_kaggle_submission.ipynb`
bilan bir xil naqsh).

In [7]:
feature_cols = get_numeric_feature_cols(train, cfg)
esr = cfg.models["lgbm"]["early_stopping_rounds"]

final_result = run_cv(
    lambda: build_lgbm_model(cfg), train, feature_cols, cfg, "lgbm_final",
    fit_kwargs_fn=make_lgbm_fit_kwargs(esr), save_models=True, log_progress=False,
)
print(f"Tasdiqlash: CV log_loss = {final_result.cv_log_loss:.5f} (kutilgan: 0.36288)")

test_proba_folds = [m.predict_proba(test[feature_cols]) for m in final_result.models]
test_proba_raw = np.mean(test_proba_folds, axis=0)

EPS = 1e-7
test_proba = np.clip(test_proba_raw, EPS, 1 - EPS)
test_proba = test_proba / test_proba.sum(axis=1, keepdims=True)

submission = pd.DataFrame(test_proba, columns=[f"Status_{c}" for c in CLASSES], index=test.index)
submission.index.name = cfg.data.id_col
submission = submission.reset_index()

cfg.paths.submissions_dir.mkdir(parents=True, exist_ok=True)
out_path = cfg.paths.submissions_dir / "final_lgbm.csv"
submission.to_csv(out_path, index=False)
print(f"Saqlandi: {out_path} ({submission.shape[0]} qator, {submission.shape[1]} ustun)")
submission.head()

Tasdiqlash: CV log_loss = 0.36288 (kutilgan: 0.36288)


Saqlandi: /Users/dilshodjon216/working/ml-portfolio/cirrhosis-outcomes/outputs/submissions/final_lgbm.csv (10000 qator, 4 ustun)


,id,Status_C,Status_CL,Status_D
0,15000,0.437200,0.002749,0.560052
1,15001,0.856532,0.014500,0.128967
2,15002,0.724602,0.004878,0.270520
3,15003,0.985532,0.000557,0.013910
4,15004,0.181301,0.102836,0.715863


## Yakuniy xulosalar

1. **Ensemble har doim yordam bermaydi** — bu holatda uchta model bir xil
   feature to'plamida o'qitilgan o'xshash daraxt modellari bo'lgani uchun
   ularning xatoliklari juda korrelyatsiyalangan, blend/stacking uchun
   kam "xilma-xillik" qoldirgan.
2. **Zaif modelni teng vaznda qo'shish zararli** — oddiy o'rtacha blend
   `catboost`ning yomonroq natijasi tufayli yakka eng yaxshi modeldan
   yomonroq chiqdi.
3. **Optimallashtirish o'zi "catboost foydasiz" degan xulosaga keldi** —
   vazn deyarli 0 chiqdi, bu qo'lda sinash o'rniga avtomatik tasdiqlash edi.
4. **Stacking eng ishonchsiz usul bo'lib chiqdi** — kichik ma'lumot +
   ko'p korrelyatsiyalangan feature + qo'shimcha CV qatlami = ortiqcha shovqin.
5. **Yakuniy qaror — soddalik g'alaba qozondi:** yakka, sozlangan LightGBM
   model (`CV log_loss = 0.36288`) yakuniy yechim sifatida tanlandi.
   Yakuniy bashorat `outputs/submissions/final_lgbm.csv` ga saqlandi.

**Keyingi qadam (Faza 8):** production tozalash — `src/predict.py`,
testlar, CI, `README.md`.